# Fase 1 y 2 — Datos sintéticos y diálogos
**POC Speech Analytics · Retención de desafiliaciones · Seguro vehicular (caso ficticio)**

Este notebook corre en Google Colab. Orden:
1. Cargar el proyecto
2. Generar las tablas operativas y la verdad de terreno
3. Configurar la clave de OpenAI (con *Secrets* de Colab, nunca pegada en el código)
4. Probar con 5 diálogos y revisarlos a mano
5. Generar la muestra completa

### **1. Cargar el proyecto**
Sube `speech-analytics-retencion-autos.zip` con el ícono de carpeta de la izquierda y ejecuta la celda. Si ya lo subiste a GitHub, usa la línea comentada.

In [ ]:
# !git clone https://github.com/<tu-usuario>/speech-analytics-retencion-autos.git
!unzip -q -o speech-analytics-retencion-autos.zip
%cd speech-analytics-retencion-autos
!pip install -q -r requirements.txt

### **2. Generar datos sintéticos y revisar KPIs**
Toma ~5 segundos. La semilla fija hace que siempre salgan los mismos datos.

In [ ]:
!python src/generar_datos.py --salida data/raw --semilla 42
!python src/revisar_kpis.py --datos data/raw

### **3. Clave de OpenAI**
En Colab: ícono de llave 🔑 (Secrets) → *Add new secret* → nombre `OPENAI_API_KEY` → pega tu clave → activa *Notebook access*.

**Ojo:** nunca escribas la clave en una celda; si subes el notebook a GitHub quedaría expuesta.

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from openai import OpenAI
modelos = sorted(m.id for m in OpenAI().models.list().data)
print([m for m in modelos if any(k in m for k in ("luna", "mini", "transcribe", "tts", "whisper"))])

Elige el modelo de texto más barato disponible en tu cuenta (por defecto el script usa `gpt-6-luna`). Si no aparece, cambia la variable `MODELO`.

In [ ]:
MODELO = "gpt-6-luna"   # cámbialo si tu cuenta no lo tiene

### **4. Prueba piloto con 5 diálogos**
Antes de gastar en 400, revisa que el diálogo **cumpla la ficha**: si la ficha dice que el asesor no indaga, no debe indagar. Esta revisión manual es parte del método.

In [ ]:
!python src/generar_dialogos.py --n 5 --modelo $MODELO --salida data/dialogos_piloto

In [ ]:
import json, pandas as pd, sys
sys.path.insert(0, "src")
from generar_dialogos import construir_ficha

muestra = pd.read_csv("data/dialogos_piloto/muestra.csv").set_index("llamada_id")
dialogos = [json.loads(l) for l in open("data/dialogos_piloto/dialogos.jsonl", encoding="utf-8")]

d = dialogos[0]
print(construir_ficha(muestra.loc[d["llamada_id"]].rename(d["llamada_id"])))
print("\n" + "=" * 80 + "\n")
for t in d["turnos"]:
    print(f"{t['hablante'].upper():8} {t['texto']}")

### **5. Generación completa (~400 llamadas)**
Si se corta, vuelve a ejecutar: el script retoma donde quedó. Al final imprime los tokens usados para que calcules el costo real.

In [ ]:
!python src/generar_dialogos.py --n 400 --modelo $MODELO --salida data/dialogos --hilos 6

In [ ]:
dialogos = [json.loads(l) for l in open("data/dialogos/dialogos.jsonl", encoding="utf-8")]
ok = [d for d in dialogos if d.get("turnos")]
palabras = pd.Series([sum(len(t["texto"].split()) for t in d["turnos"]) for d in ok])
print(f"Diálogos válidos: {len(ok)} de {len(dialogos)}")
print(palabras.describe().round(0))

### **6. Guardar en Google Drive**
Colab borra los archivos al cerrar la sesión.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/speech-analytics-retencion-autos
!cp -r data /content/drive/MyDrive/speech-analytics-retencion-autos/